# MLflow Part 1 — FIFA 22 Players Dataset

## What is MLflow?

**MLflow** is an open-source platform for managing the full machine learning lifecycle. It solves a common problem in ML: experiments are hard to reproduce, compare, and share because parameters, metrics, and model files end up scattered across notebooks, scripts, and local folders.

MLflow organises everything around four core concepts:

| Concept | What it is |
|---|---|
| **Tracking** | Records parameters, metrics, and artifacts for every training run |
| **Projects** | Packages code for reproducible execution |
| **Models** | A standard format to package and deploy trained models |
| **Registry** | A centralised store to version, stage, and manage models |

---

## Where do runs get stored? DagsHub

By default MLflow stores everything locally. To share experiments with a team and get a hosted UI, we use **[DagsHub](https://dagshub.com)** as the remote tracking server.

DagsHub is a collaborative platform for ML projects that integrates Git, DVC, and MLflow in one place. It provides a free hosted MLflow tracking server for every repository, so you can see all your runs, compare metrics, and browse artifacts online without running any infrastructure yourself.

> **Create a free account at [https://dagshub.com/user/sign_up](https://dagshub.com/user/sign_up)**

Once you have an account, create a repository and fill in `DAGSHUB_USER` and `DAGSHUB_REPO` in section 3.

---

## What we will do

We use the FIFA 22 players dataset to:
1. Load and prepare the data
2. Train a few ML models to predict a player's `overall` rating
3. Track experiments with MLflow using DagsHub as the remote tracking server

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv("data/players_22.csv", low_memory=False)

# Keep only numeric skill attributes as features
feature_cols = [
    "pace", "shooting", "passing", "dribbling", "defending", "physic",
    "age", "height_cm", "weight_kg", "weak_foot", "skill_moves", "international_reputation"
]
target_col = "overall"

data = df[feature_cols + [target_col]].dropna()
print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 3. Connect to DagsHub

**`dagshub.init()`** points MLflow's tracking URI to your DagsHub repository instead of a local folder. Without this, runs are only stored on your machine and can't be shared or compared remotely.

**`mlflow.set_experiment()`** groups all runs under a named experiment. Every run we create below will appear under `fifa22-overall-prediction` in the DagsHub UI, keeping them organised and separate from other projects.

In [ ]:
DAGSHUB_USER = "atdepo"
DAGSHUB_REPO = "SE4AI_2026_MLFlow_Lab"

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("fifa22-overall-prediction-v1")

## 4. Models

`all_results` will collect each model's predictions for the final comparison.

In [ ]:
import matplotlib.pyplot as plt

all_results = {}

### 4.1 Ridge Regression

Each iteration of the hyperparameter sweep opens a **`mlflow.start_run()`** context. Everything inside that block — params, metrics, and the model file — is recorded as a single, self-contained **run**:

- **`mlflow.log_params()`** — saves the hyperparameters used (e.g. `alpha`). Needed to remember *what configuration* produced a given result.
- **`mlflow.log_metrics()`** — saves the evaluation scores (MAE, R²). Needed to compare *how well* each configuration performed.
- **`mlflow.sklearn.log_model()`** — serialises and uploads the trained model as an artifact. Needed to be able to reload and serve the exact model later without retraining.

In [ ]:
ridge_runs = []

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    with mlflow.start_run(run_name=f"Ridge_alpha={alpha}"):
        model = Ridge(alpha=alpha)
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)

        mae = mean_absolute_error(y_test, preds)
        r2  = r2_score(y_test, preds)

        mlflow.log_params({"model": "Ridge", "alpha": alpha})
        mlflow.log_metrics({"mae": mae, "r2": r2})
        mlflow.sklearn.log_model(model, name="model")

        ridge_runs.append({"alpha": alpha, "mae": mae, "r2": r2, "preds": preds, "model": model})
        print(f"Ridge alpha={alpha:<6} | MAE: {mae:.3f} | R2: {r2:.3f}")

best_ridge = min(ridge_runs, key=lambda x: x["mae"])
all_results["Ridge"] = best_ridge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

alphas = [r["alpha"] for r in ridge_runs]
axes[0].plot(alphas, [r["mae"] for r in ridge_runs], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("MAE")
axes[0].set_title("Ridge — MAE vs alpha")

axes[1].scatter(y_test, best_ridge["preds"], alpha=0.3, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_title(f"Ridge best (alpha={best_ridge['alpha']}) | MAE={best_ridge['mae']:.2f}  R²={best_ridge['r2']:.3f}")
axes[1].set_xlabel("Actual overall")
axes[1].set_ylabel("Predicted overall")

plt.tight_layout()
plt.show()

### 4.2 Random Forest

Same MLflow pattern as above — one run per `max_depth` value. Because every run logs the same metric keys (`mae`, `r2`), DagsHub lets you plot all runs on the same chart and immediately see how tree depth affects performance.

In [ ]:
rf_runs = []

for max_depth in [3, 5, 10, 15, 20]:
    with mlflow.start_run(run_name=f"RandomForest_depth={max_depth}"):
        model = RandomForestRegressor(n_estimators=100, max_depth=max_depth, random_state=42)
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)

        mae = mean_absolute_error(y_test, preds)
        r2  = r2_score(y_test, preds)

        mlflow.log_params({"model": "RandomForest", "n_estimators": 100, "max_depth": max_depth})
        mlflow.log_metrics({"mae": mae, "r2": r2})
        mlflow.sklearn.log_model(model, name="model")

        rf_runs.append({"max_depth": max_depth, "mae": mae, "r2": r2, "preds": preds, "model": model})
        print(f"RandomForest max_depth={max_depth:<3} | MAE: {mae:.3f} | R2: {r2:.3f}")

best_rf = min(rf_runs, key=lambda x: x["mae"])
all_results["RandomForest"] = best_rf

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

depths = [r["max_depth"] for r in rf_runs]
axes[0].plot(depths, [r["mae"] for r in rf_runs], marker="o")
axes[0].set_xlabel("max_depth")
axes[0].set_ylabel("MAE")
axes[0].set_title("RandomForest — MAE vs max_depth")

axes[1].scatter(y_test, best_rf["preds"], alpha=0.3, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_title(f"RandomForest best (depth={best_rf['max_depth']}) | MAE={best_rf['mae']:.2f}  R²={best_rf['r2']:.3f}")
axes[1].set_xlabel("Actual overall")
axes[1].set_ylabel("Predicted overall")

plt.tight_layout()
plt.show()

### 4.3 XGBoost

Same MLflow pattern, sweeping `learning_rate`. Note that all three model families share the same experiment — this is intentional: MLflow lets you compare runs across different model types in a single view, which is how you pick the best overall model.

In [ ]:
xgb_runs = []

for learning_rate in [0.01, 0.05, 0.1, 0.2, 0.3]:
    with mlflow.start_run(run_name=f"XGBoost_lr={learning_rate}"):
        model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=learning_rate, random_state=42, verbosity=0)
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)

        mae = mean_absolute_error(y_test, preds)
        r2  = r2_score(y_test, preds)

        mlflow.log_params({"model": "XGBoost", "n_estimators": 100, "max_depth": 6, "learning_rate": learning_rate})
        mlflow.log_metrics({"mae": mae, "r2": r2})
        mlflow.sklearn.log_model(model, name="model")

        xgb_runs.append({"learning_rate": learning_rate, "mae": mae, "r2": r2, "preds": preds, "model": model})
        print(f"XGBoost lr={learning_rate:<4} | MAE: {mae:.3f} | R2: {r2:.3f}")

best_xgb = min(xgb_runs, key=lambda x: x["mae"])
all_results["XGBoost"] = best_xgb

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

lrs = [r["learning_rate"] for r in xgb_runs]
axes[0].plot(lrs, [r["mae"] for r in xgb_runs], marker="o")
axes[0].set_xlabel("learning_rate")
axes[0].set_ylabel("MAE")
axes[0].set_title("XGBoost — MAE vs learning_rate")

axes[1].scatter(y_test, best_xgb["preds"], alpha=0.3, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_title(f"XGBoost best (lr={best_xgb['learning_rate']}) | MAE={best_xgb['mae']:.2f}  R²={best_xgb['r2']:.3f}")
axes[1].set_xlabel("Actual overall")
axes[1].set_ylabel("Predicted overall")

plt.tight_layout()
plt.show()

## 5. Overall Comparison

In [ ]:
summary = pd.DataFrame(
    {"MAE": {k: v["mae"] for k, v in all_results.items()},
     "R2":  {k: v["r2"]  for k, v in all_results.items()}}
).sort_values("MAE")

display(summary)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, res) in zip(axes, all_results.items()):
    ax.scatter(y_test, res["preds"], alpha=0.3, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    ax.set_title(f"{name}\nMAE={res['mae']:.2f}  R²={res['r2']:.3f}")
    ax.set_xlabel("Actual overall")
    ax.set_ylabel("Predicted overall")
plt.tight_layout()
plt.show()

## 6. Log Artifacts

Metrics and params are just numbers. **Artifacts** are any file you want to attach to a run — plots, CSVs, reports, config files.

- **`mlflow.log_figure()`** — uploads a matplotlib figure directly without saving it to disk first. Useful for plots you want to inspect later in the UI (e.g. feature importances, residual plots).
- **`mlflow.log_artifact()`** — uploads any local file. Here we attach a CSV of predictions vs actuals so anyone can audit the model outputs without rerunning the notebook.
- **`registered_model_name=`** inside `log_model()` — registers the model in the **Model Registry** in one step.

In [ ]:
import tempfile, os

MODEL_NAME     = "fifa22-overall-xgboost"
best_xgb_model = best_xgb["model"]
best_lr        = best_xgb["learning_rate"]

with mlflow.start_run(run_name=f"XGBoost_artifacts_lr={best_lr}") as run:
    artifact_run_id = run.info.run_id

    preds = best_xgb_model.predict(X_test_scaled)
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)

    mlflow.log_params({"model": "XGBoost", "n_estimators": 100, "max_depth": 6, "learning_rate": best_lr})
    mlflow.log_metrics({"mae": mae, "r2": r2})

    # register the model directly from log_model — avoids artifact_path lookup issues
    mlflow.sklearn.log_model(best_xgb_model, name="model", registered_model_name=MODEL_NAME)

    # --- artifact 1: feature importance plot ---
    fig, ax = plt.subplots(figsize=(7, 4))
    importances = best_xgb_model.feature_importances_
    ax.barh(feature_cols, importances)
    ax.set_xlabel("Importance")
    ax.set_title("XGBoost — Feature Importances")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.show()

    # --- artifact 2: predictions CSV ---
    with tempfile.TemporaryDirectory() as tmp:
        csv_path = os.path.join(tmp, "predictions.csv")
        pd.DataFrame({"actual": y_test.values, "predicted": preds}).to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path)

    print(f"Model registered as '{MODEL_NAME}' from run {artifact_run_id}")

## 7. Model Registry

The **Model Registry** is a centralised catalogue of your models. While a *run* is a snapshot of an experiment, a *registered model* is a versioned, named entity you can promote through a lifecycle.

**`MlflowClient`** is the low-level API for interacting with the tracking server programmatically — querying runs, managing model versions, setting aliases, etc.

**`search_model_versions()`** retrieves all versions of a registered model so we can find the latest one to promote.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = max(versions, key=lambda v: int(v.version))
print(f"Latest version of '{MODEL_NAME}': {latest.version}")

**`set_registered_model_alias()`** assigns a human-readable label (e.g. `"staging"`, `"production"`) to a specific model version. Aliases replace the old stage system (deprecated in MLflow 2.9+) and let downstream code always load the model behind an alias without hardcoding a version number — you just update the alias when you promote a new version.

In [ ]:
# Modern MLflow uses aliases instead of stages
client.set_registered_model_alias(MODEL_NAME, "staging", latest.version)
print(f"Version {latest.version} → alias 'staging'")

client.set_registered_model_alias(MODEL_NAME, "production", latest.version)
print(f"Version {latest.version} → alias 'production'")

## 8. Load & Serve

**`mlflow.sklearn.load_model(f"models:/<name>@<alias>")`** pulls the model directly from the registry using its alias. This is the key benefit of the registry: consuming code never needs to know the run ID or version number — it just asks for `@production` and always gets the right model. Swapping in a new version is a one-line alias update, with no changes needed in the serving code.

In [ ]:
loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@production")

sample = X_test.sample(5, random_state=0)
sample_scaled = scaler.transform(sample)

predictions = loaded_model.predict(sample_scaled)

result = sample.copy()
result["predicted_overall"] = predictions.round(1)
result["actual_overall"]    = y_test.loc[sample.index].values
result